#### 1. 재무데이터 입력하기

In [29]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Financial Modeling Prep API - 정리된 데이터 수집 스크립트
분기별 매출 데이터 + 월별 시가총액 데이터 (월말 날짜 통일)
"""

# ==============================================
# 필수 라이브러리 import
# ==============================================
import requests
import pandas as pd
import numpy as np
import calendar

import time
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
from DATA.stock_invest_function import *

from datetime import datetime, timedelta

# plotting 설정
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

# ==============================================
# 설정값들
# ==============================================

# API 키 설정
apikey = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'
API_KEY = apikey

# 파라미터 설정
tic_name = 'ANET'
hs_code = '851762'
item_name = 'PSR'
st_date = '2010-01-01'
end_date = '2025-07-31'
today_date = pd.to_datetime(datetime.today().date())
USE_EXOGENOUS = True

# 테스트용 기업
TICKERS = ['MMM']
ticker = TICKERS[0]
MAX_RETRIES = 2
REQUEST_DELAY = 0.3
RETRY_DELAY = 1.0

# ==============================================
# 유틸리티 함수들
# ==============================================

def test_api_connection():
    """API 연결 테스트"""
    test_url = f"https://financialmodelingprep.com/api/v3/income-statement/AAPL"
    test_params = {'limit': 1, 'apikey': API_KEY, 'period': 'quarter'}

    try:
        response = requests.get(test_url, params=test_params, timeout=10)

        if response.status_code == 401:
            return False, "API 키가 유효하지 않습니다."
        elif response.status_code == 429:
            return False, "API 요청 한도를 초과했습니다."
        elif response.status_code != 200:
            return False, f"API 오류: {response.status_code}"

        data = response.json()
        if isinstance(data, dict) and 'Error Message' in data:
            return False, f"API 오류: {data['Error Message']}"
        elif not data:
            return False, "API에서 빈 응답을 받았습니다."

        return True, f"API 연결 성공"

    except Exception as e:
        return False, f"API 연결 실패: {str(e)}"

def convert_to_month_end(date_str):
    """날짜를 해당 월의 월말로 변환"""
    try:
        # 문자열이나 Timestamp를 datetime으로 변환
        if isinstance(date_str, str):
            date_obj = pd.to_datetime(date_str)
        else:
            date_obj = date_str

        # 해당 월의 마지막 날 계산
        year = date_obj.year
        month = date_obj.month
        last_day = calendar.monthrange(year, month)[1]

        # 월말 날짜 생성
        month_end = datetime(year, month, last_day)
        return month_end

    except Exception as e:
        print(f"날짜 변환 오류: {date_str} -> {e}")
        return None

def add_revenue_ttm(df):
    """
    분기별 매출 데이터에 TTM(최근 4분기 합계) 컬럼 추가
    """
    df_copy = df.copy()
    df_copy = df_copy.sort_values(['ticker', 'date'])

    # TTM 계산을 위한 빈 리스트
    ttm_values = []

    # 티커별로 TTM 계산
    for ticker in df_copy['ticker'].unique():
        ticker_data = df_copy[df_copy['ticker'] == ticker].copy()
        ticker_data = ticker_data.sort_values('date')

        # rolling sum으로 최근 4분기 합계 계산
        ticker_data['revenue_ttm'] = ticker_data['revenue'].rolling(window=4, min_periods=1).sum()

        ttm_values.extend(ticker_data['revenue_ttm'].tolist())

    df_copy['revenue_ttm'] = ttm_values
    return df_copy

def fetch_revenue_data(ticker, retry_count=0):
    """분기별 매출 데이터 수집"""
    url = f"https://financialmodelingprep.com/api/v3/income-statement/{ticker}"
    params = {'limit': 200, 'apikey': API_KEY, 'period': 'quarter'}

    try:
        response = requests.get(url, params=params, timeout=30)

        if response.status_code != 200:
            return None, f"HTTP {response.status_code}"

        data = response.json()

        if isinstance(data, dict) and 'Error Message' in data:
            return None, f"API 오류: {data['Error Message']}"

        if not data:
            return None, "데이터 없음"

        return data, None

    except Exception as e:
        return None, f"오류: {str(e)}"


def check_variables():
    """변수 존재 확인 함수"""
    print("현재 생성된 DataFrame 변수들:")

    vars_to_check = ['revenue_df', 'monthly_df', 'revenue_df_with_ttm']
    for var_name in vars_to_check:
        if var_name in globals():
            df = globals()[var_name]
            if isinstance(df, pd.DataFrame) and not df.empty:
                print(f"✅ {var_name}: {df.shape}")
            else:
                print(f"⚠️ {var_name}: 빈 DataFrame")
        else:
            print(f"❌ {var_name}: 존재하지 않음")

# ==============================================
# API 키 확인 및 연결 테스트
# ==============================================

# API 키 유효성 검사
if not API_KEY or API_KEY == "YOUR_API_KEY_HERE":
    print("❌ 유효한 API 키를 설정해주세요!")
    print("현재 API_KEY:", API_KEY[:10] + "..." if API_KEY else "None")
else:
    print(f"✅ API 키 설정됨: {API_KEY[:10]}...")

print("🔍 API 연결 테스트 중...")
api_ok, api_message = test_api_connection()
print(api_message)

if not api_ok:
    print("⚠️ API 연결에 실패했지만 계속 진행합니다. API 키와 네트워크 상태를 확인해주세요.")

print(f"🚀 경량 데이터 수집 시작")
print(f"📊 대상 기업: {len(TICKERS)}개 - {TICKERS}")
print(f"📈 수집 데이터: 분기별 매출 + 월별 시가총액")
print("=" * 70)

# ==============================================
# 1. 분기별 매출 데이터 수집
# ==============================================
print("📈 Fetching quarterly revenue data...")
all_revenue_data = []
revenue_successful_tickers = []

for ticker in tqdm(TICKERS, desc="Revenue"):
    revenue_data, error = fetch_revenue_data(ticker)

    if revenue_data is None:
        print(f"   ❌ {ticker}: {error}")
        continue

    for item in revenue_data:
        all_revenue_data.append({
            'ticker': ticker,
            'date': item.get('date', ''),
            'calendar_year': item.get('calendarYear', ''),
            'period': item.get('period', ''),
            'revenue': item.get('revenue', 0) if item.get('revenue') is not None else 0,
            'revenue_billions': round((item.get('revenue', 0) or 0) / 1_000_000_000, 2),
            'gross_profit': item.get('grossProfit', 0) if item.get('grossProfit') is not None else 0,
            'gross_margin': round(((item.get('grossProfit', 0) or 0) / (item.get('revenue', 1) or 1)) * 100, 2) if (item.get('revenue') or 0) > 0 else 0,
        })

    revenue_successful_tickers.append(ticker)
    print(f"   ✅ {ticker}: {len([d for d in all_revenue_data if d['ticker'] == ticker])}개 분기")
    time.sleep(REQUEST_DELAY)

# DataFrame 생성
revenue_df = pd.DataFrame(all_revenue_data) if all_revenue_data else pd.DataFrame()

# 3. revenue_df가 이미 존재한다고 가정하고 TTM 추가
print("📊 TTM 매출 컬럼 추가 중...")

if 'revenue_df' in globals() and not revenue_df.empty:
    # TTM 컬럼 추가
    revenue_df_with_ttm = add_revenue_ttm(revenue_df)

    # TTM을 십억 단위로 변환
    revenue_df_with_ttm['revenue_ttm_billions'] = revenue_df_with_ttm['revenue_ttm'] / 1_000_000_000

    # 월말 날짜 컬럼 추가
    print("📅 revenue_df_with_ttm에 월말 날짜 컬럼 추가 중...")
    revenue_df_with_ttm['date_month_end'] = revenue_df_with_ttm['date'].apply(convert_to_month_end)

    print(f"✅ revenue_df_with_ttm 생성 완료: {revenue_df_with_ttm.shape}")

    # TTM 데이터 샘플 확인
    print(f"\n📋 TTM 매출 데이터 샘플 (최신 5개):")
    sample_cols = ['ticker', 'date', 'date_month_end', 'period', 'revenue_billions', 'revenue_ttm_billions']
    available_cols = [col for col in sample_cols if col in revenue_df_with_ttm.columns]
    print(revenue_df_with_ttm[available_cols].tail(5).to_string(index=False))

else:
    print("❌ revenue_df가 존재하지 않습니다. 먼저 revenue_df를 생성해주세요.")

    # revenue_df가 없는 경우를 위한 샘플 생성 코드
    print("\n revenue_df 생성이 필요한 경우, 다음과 같은 구조여야 합니다:")
    print("필수 컬럼: ['ticker', 'date', 'period', 'revenue', 'revenue_billions', ...]")

# 4. revenue_df_with_ttm 확인 함수
def check_revenue_ttm():
    """revenue_df_with_ttm 존재 및 구조 확인"""
    if 'revenue_df_with_ttm' in globals():
        df = revenue_df_with_ttm
        print(f"✅ revenue_df_with_ttm 존재: {df.shape}")
        print(f"   컬럼: {list(df.columns)}")
        print(f"   날짜 범위: {df['date'].min()} ~ {df['date'].max()}")

        if 'revenue_ttm' in df.columns:
            print(f"   TTM 데이터: 평균 ${df['revenue_ttm'].mean()/1e9:.2f}B")
        if 'date_month_end' in df.columns:
            print(f"   월말 날짜: 변환 완료")
    else:
        print("❌ revenue_df_with_ttm이 존재하지 않습니다.")


if not revenue_df.empty:
    # 날짜 컬럼 처리
    revenue_df['date'] = pd.to_datetime(revenue_df['date'])
    revenue_df = revenue_df.sort_values(['ticker', 'date'], ascending=[True, True])

    # 월말 날짜 컬럼 추가
    print("📅 revenue_df에 월말 날짜 컬럼 추가 중...")
    revenue_df['date_month_end'] = revenue_df['date'].apply(convert_to_month_end)

    print(f"✅ revenue_df 생성 완료: {len(revenue_df)} 레코드")

    # TTM 컬럼 추가
    print("📊 TTM 매출 컬럼 추가 중...")
    revenue_df_with_ttm = add_revenue_ttm(revenue_df)
    revenue_df_with_ttm['revenue_ttm_billions'] = revenue_df_with_ttm['revenue_ttm'] / 1_000_000_000

    # TTM DataFrame에도 월말 날짜 컬럼 추가
    print("📅 revenue_df_with_ttm에 월말 날짜 컬럼 추가 중...")
    revenue_df_with_ttm['date_month_end'] = revenue_df_with_ttm['date'].apply(convert_to_month_end)

    print(f"✅ TTM 컬럼 추가 완료")

    # TTM 데이터 샘플 확인
    print(f"\n📋 TTM 매출 데이터 샘플 (최신 5개):")
    ttm_sample = revenue_df_with_ttm[['ticker', 'date', 'date_month_end', 'period', 'revenue_billions', 'revenue_ttm_billions']].tail(5)
    print(ttm_sample.to_string(index=False))

✅ API 키 설정됨: hT0gAk87j9...
🔍 API 연결 테스트 중...
API 연결 성공
🚀 경량 데이터 수집 시작
📊 대상 기업: 1개 - ['MMM']
📈 수집 데이터: 분기별 매출 + 월별 시가총액
📈 Fetching quarterly revenue data...


Revenue:   0%|          | 0/1 [00:00<?, ?it/s]

   ✅ MMM: 160개 분기


Revenue: 100%|██████████| 1/1 [00:01<00:00,  1.25s/it]

📊 TTM 매출 컬럼 추가 중...
📅 revenue_df_with_ttm에 월말 날짜 컬럼 추가 중...
✅ revenue_df_with_ttm 생성 완료: (160, 11)

📋 TTM 매출 데이터 샘플 (최신 5개):
ticker       date date_month_end period  revenue_billions  revenue_ttm_billions
   MMM 2024-06-30     2024-06-30     Q2              6.25                24.543
   MMM 2024-09-30     2024-09-30     Q3              6.29                24.567
   MMM 2024-12-31     2024-12-31     Q4              6.01                24.575
   MMM 2025-03-31     2025-03-31     Q1              5.95                24.513
   MMM 2025-06-30     2025-06-30     Q2              6.34                24.602
📅 revenue_df에 월말 날짜 컬럼 추가 중...
✅ revenue_df 생성 완료: 160 레코드
📊 TTM 매출 컬럼 추가 중...
📅 revenue_df_with_ttm에 월말 날짜 컬럼 추가 중...
✅ TTM 컬럼 추가 완료

📋 TTM 매출 데이터 샘플 (최신 5개):
ticker       date date_month_end period  revenue_billions  revenue_ttm_billions
   MMM 2024-06-30     2024-06-30     Q2              6.25                24.543
   MMM 2024-09-30     2024-09-30     Q3              6.29                24

#### 2. 시가총액 데이터 입력

In [30]:
def fetch_market_data_yearly(ticker, start_year=2010):
    """연도별로 세분화해서 데이터 수집"""
    all_data = []
    current_year = datetime.now().year

    for year in range(start_year, current_year + 1):
        start_date_str = f"{year}-01-01"
        end_date_str = f"{year}-12-31"

        url = f"https://financialmodelingprep.com/api/v3/historical-market-capitalization/{ticker}"
        params = {'from': start_date_str, 'to': end_date_str, 'apikey': API_KEY}

        print(f"{year}년 데이터 수집 중...")

        try:
            response = requests.get(url, params=params, timeout=30)
            if response.status_code == 200:
                data = response.json()
                if data and isinstance(data, list):
                    all_data.extend(data)
                    print(f"  {year}년: {len(data)}개 데이터")
                else:
                    print(f"  {year}년: 데이터 없음")
            else:
                print(f"  {year}년: HTTP {response.status_code}")

            time.sleep(0.3)  # API 제한 고려

        except Exception as e:
            print(f"  {year}년 오류: {str(e)}")

    print(f"총 수집 데이터: {len(all_data)}개")
    return all_data if all_data else None, None


def process_daily_to_monthly_market_data(daily_data):
    """일별 시가총액 데이터를 월말 기준으로 변환"""
    if not daily_data:
        print("데이터가 없습니다.")
        return pd.DataFrame()

    print(f"일별 데이터 처리 시작: {len(daily_data)}개 레코드")

    # DataFrame 생성
    df = pd.DataFrame(daily_data)

    # 날짜 컬럼 처리
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date')

    # 연월 컬럼 추가 (그룹화용)
    df['year_month'] = df['date'].dt.to_period('M')

    print(f"날짜 범위: {df['date'].min().strftime('%Y-%m-%d')} ~ {df['date'].max().strftime('%Y-%m-%d')}")
    print(f"총 월수: {df['year_month'].nunique()}개월")

    # 각 월의 마지막 날짜 데이터만 추출
    monthly_data = []

    for year_month in df['year_month'].unique():
        month_data = df[df['year_month'] == year_month]

        # 해당 월의 가장 마지막 날짜 데이터 선택
        last_day_data = month_data.loc[month_data['date'].idxmax()]

        monthly_data.append({
            'ticker': last_day_data.get('symbol', 'UNKNOWN'),  # API에서는 'symbol'로 올 수 있음
            'date': last_day_data['date'],
            'market_cap': last_day_data['marketCap'],
            'market_cap_billions': round(last_day_data['marketCap'] / 1_000_000_000, 2),
            'year_month': year_month
        })

    # DataFrame 생성
    monthly_df = pd.DataFrame(monthly_data)

    print(f"월별 데이터 생성 완료: {len(monthly_df)}개 레코드")

    return monthly_df

# 연도별 데이터 수집 및 월별 변환 통합 함수
def fetch_and_process_yearly_market_data(ticker, start_year=2010):
    """연도별 수집 후 월별로 변환"""
    print(f"=== {ticker} 연도별 데이터 수집 및 월별 변환 ===")

    # 연도별 데이터 수집
    data, error = fetch_market_data_yearly(ticker, start_year)

    if not data:
        print(f"데이터 수집 실패: {error}")
        return pd.DataFrame()

    # 월별 데이터로 변환
    monthly_df = process_daily_to_monthly_market_data(data)

    if monthly_df.empty:
        print("월별 변환 실패")
        return pd.DataFrame()

    # convert_to_month_end 적용
    print("월말 날짜로 표준화 중...")
    monthly_df['date_month_end'] = monthly_df['date'].apply(convert_to_month_end)

    # 불필요한 컬럼 제거
    marketcap_df = monthly_df[['ticker', 'date', 'date_month_end', 'market_cap', 'market_cap_billions']].copy()

    print(f"최종 marketcap_df 생성 완료: {marketcap_df.shape}")
    print(f"날짜 범위: {marketcap_df['date_month_end'].min().strftime('%Y-%m-%d')} ~ {marketcap_df['date_month_end'].max().strftime('%Y-%m-%d')}")

    # 샘플 데이터 출력
    print("\n월말 시가총액 데이터 샘플 (최신 5개):")
    print(marketcap_df.tail(5)[['ticker', 'date', 'date_month_end', 'market_cap_billions']].to_string(index=False))

    return marketcap_df

# 실행 코드
print("AAPL 시가총액 데이터 수집 및 월별 변환 시작...")

# 연도별 데이터 수집 후 월별 변환
marketcap_df = fetch_and_process_yearly_market_data(ticker, 2010)

AAPL 시가총액 데이터 수집 및 월별 변환 시작...
=== MMM 연도별 데이터 수집 및 월별 변환 ===
2010년 데이터 수집 중...
  2010년: 252개 데이터
2011년 데이터 수집 중...
  2011년: 252개 데이터
2012년 데이터 수집 중...
  2012년: 250개 데이터
2013년 데이터 수집 중...
  2013년: 252개 데이터
2014년 데이터 수집 중...
  2014년: 252개 데이터
2015년 데이터 수집 중...
  2015년: 252개 데이터
2016년 데이터 수집 중...
  2016년: 252개 데이터
2017년 데이터 수집 중...
  2017년: 251개 데이터
2018년 데이터 수집 중...
  2018년: 251개 데이터
2019년 데이터 수집 중...
  2019년: 252개 데이터
2020년 데이터 수집 중...
  2020년: 253개 데이터
2021년 데이터 수집 중...
  2021년: 252개 데이터
2022년 데이터 수집 중...
  2022년: 251개 데이터
2023년 데이터 수집 중...
  2023년: 250개 데이터
2024년 데이터 수집 중...
  2024년: 252개 데이터
2025년 데이터 수집 중...
  2025년: 174개 데이터
총 수집 데이터: 3948개
일별 데이터 처리 시작: 3948개 레코드
날짜 범위: 2010-01-04 ~ 2025-09-12
총 월수: 189개월
월별 데이터 생성 완료: 189개 레코드
월말 날짜로 표준화 중...
최종 marketcap_df 생성 완료: (189, 5)
날짜 범위: 2010-01-31 ~ 2025-09-30

월말 시가총액 데이터 샘플 (최신 5개):
ticker       date date_month_end  market_cap_billions
   MMM 2025-05-30     2025-05-31                80.20
   MMM 2025-06-30     2025-06-30            

#### 4. PSR 데이터 측정

In [31]:
# revenue_df_with_ttm에서 필요한 컬럼 추출
ttm_data = revenue_df_with_ttm[['ticker', 'date_month_end', 'revenue_billions', 'revenue_ttm_billions']].copy()

# monthly_df에서 필요한 컬럼 추출
market_data = marketcap_df[['ticker', 'date_month_end', 'market_cap_billions']].copy()

# 데이터 병합 (left join)
merged_data = pd.merge(
    market_data,
    ttm_data,
    on=['ticker', 'date_month_end'],
    how='left'
)

# 종목별, 날짜별 정렬
merged_data = merged_data.sort_values(['ticker', 'date_month_end']).reset_index(drop=True)

print("   2. Forward fill 적용 (limit=3)...")

# 종목별로 그룹화하여 forward fill 적용
merged_data['revenue_ttm_billions'] = merged_data.groupby('ticker')['revenue_ttm_billions'].ffill(limit=3)
merged_data['revenue_billions'] = merged_data['revenue_billions'].ffill(limit=3)
print("   3. 결측치 제거...")

# 병합 전 레코드 수
records_before_dropna = len(merged_data)

# 결측치 제거
merged_data = merged_data.dropna(subset=['revenue_ttm_billions']).reset_index(drop=True)


# ==============================================
# TTM Shift 및 월별 PSR 계산
# ==============================================

print("📊 TTM 데이터 shift 및 PSR 계산 중...")

# revenue_ttm_billions를 2개월 뒤로 shift (종목별로)
merged_data['revenue_ttm_shift'] = merged_data.groupby('ticker')['revenue_ttm_billions'].shift(2)

print("   ✅ revenue_ttm_billions를 2개월 뒤로 shift 완료")

# 월별 PSR_ttm 계산 (시가총액 / TTM 매출)
merged_data['PSR_ttm'] = merged_data['market_cap_billions'] / merged_data['revenue_ttm_shift']

print("   ✅ PSR_ttm 계산 완료 (market_cap_billions / revenue_ttm_shift)")

# shift로 인한 NaN 값 제거
records_before_nan_removal = len(merged_data)
merged_data = merged_data.dropna(subset=['revenue_ttm_shift', 'PSR_ttm']).reset_index(drop=True)
records_after_nan_removal = len(merged_data)

print(f"   ✅ NaN 값 제거 완료")
print(f"     - NaN 제거 전: {records_before_nan_removal:,}개 레코드")
print(f"     - NaN 제거 후: {records_after_nan_removal:,}개 레코드")
print(f"     - 제거된 레코드: {records_before_nan_removal - records_after_nan_removal:,}개")

   2. Forward fill 적용 (limit=3)...
   3. 결측치 제거...
📊 TTM 데이터 shift 및 PSR 계산 중...
   ✅ revenue_ttm_billions를 2개월 뒤로 shift 완료
   ✅ PSR_ttm 계산 완료 (market_cap_billions / revenue_ttm_shift)
   ✅ NaN 값 제거 완료
     - NaN 제거 전: 187개 레코드
     - NaN 제거 후: 185개 레코드
     - 제거된 레코드: 2개


#### 5. 외생변수 입력

In [38]:
from sqlalchemy import create_engine

def get_hs_data(hs_code_6d, db_info):
    """
    HS Code로 무역 데이터 추출

    Parameters:
    - hs_code_6d (str): 6자리 HS Code
    - db_info (dict): 데이터베이스 연결 정보

    Returns:
    - pd.DataFrame: 추출된 데이터
    """

    try:
        # 데이터베이스 연결
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )

        # 데이터 조회
        query = f"""
        SELECT * FROM us_trade_monthly_data_with_forecast
        WHERE hs_code_6d = '{hs_code_6d}'
        ORDER BY date DESC
        """

        df = pd.read_sql(query, engine)
        engine.dispose()

        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])

        return df

    except Exception as e:
        print(f"오류: {str(e)}")
        return pd.DataFrame()

def get_latest_input_date_data(df):
    """
    DataFrame에서 input_date가 가장 최근인 데이터만 추출

    Parameters:
    - df (pd.DataFrame): 원본 데이터프레임

    Returns:
    - pd.DataFrame: 가장 최근 input_date의 데이터
    """

    # input_date 컬럼이 존재하는지 확인
    if 'input_date' not in df.columns:
        print("Error: 'input_date' 컬럼이 존재하지 않습니다.")
        return pd.DataFrame()

    # input_date를 datetime으로 변환 (이미 datetime이면 그대로)
    df_copy = df.copy()
    if not pd.api.types.is_datetime64_any_dtype(df_copy['input_date']):
        df_copy['input_date'] = pd.to_datetime(df_copy['input_date'])

    # 가장 최근 input_date 찾기
    latest_date = df_copy['input_date'].max()

    # 가장 최근 날짜의 데이터만 필터링
    latest_data = df_copy[df_copy['input_date'] == latest_date].copy()

    print(f"가장 최근 input_date: {latest_date.strftime('%Y-%m-%d')}")
    print(f"해당 날짜의 데이터: {len(latest_data):,}개")

    return latest_data

In [44]:
root_hs_code = '854232'   # 로그 변환 적용 HS Code
FORECAST_STEPS = 15                # 예측 개월 수
MIN_PERIODS = 60                   # 최소 데이터 개수(5년)

# DB 접속 정보
db_info = {
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'host': get_db_host(),
    'port': '3307',
    'database': 'investar'
}

# HS Code로 데이터 추출
export_df = get_hs_data('851762', db_info)

# 사용 예시 (export_df가 있다고 가정)
if 'export_df' in globals():
    # 가장 최근 input_date 데이터 추출
    latest_export_data = get_latest_input_date_data(export_df)
    latest_export_data = latest_export_data.sort_values('date').reset_index(drop=True)
    # 결과 확인
    if not latest_export_data.empty:
        print("\n추출된 데이터 샘플:")
        display_cols = ['hs_code_6d', 'date', 'export_value', 'input_date']
        available_cols = [col for col in display_cols if col in latest_export_data.columns]
        print(latest_export_data[available_cols].head().to_string(index=False))
else:
    print("export_df 변수가 존재하지 않습니다.")


가장 최근 input_date: 2025-09-02
해당 날짜의 데이터: 165개

추출된 데이터 샘플:
hs_code_6d       date input_date
    851762 2013-01-31 2025-09-02
    851762 2013-02-28 2025-09-02
    851762 2013-03-31 2025-09-02
    851762 2013-04-30 2025-09-02
    851762 2013-05-31 2025-09-02


In [52]:
print("데이터프레임 결합 시작...")

# 1. latest_export_data에서 필요한 컬럼 추출
export_subset = latest_export_data[['hs_code_6d', 'date', 'expDlr']].copy()

print(f"수출 데이터: {len(export_subset)}개 레코드")

# 2. latest_export_data의 date를 월말로 변환
export_subset['date'] = pd.to_datetime(export_subset['date'])
export_subset['date_month_end'] = export_subset['date'].apply(convert_to_month_end)

print("수출 데이터 월말 변환 완료")

# 3. merged_data와 결합 (left join - merged_data 기준)
final_data = pd.merge(
    export_subset[['date_month_end', 'hs_code_6d', 'expDlr']],  # 필요한 컬럼만
    merged_data,
    on='date_month_end',
    how='left'
)

print(f"데이터 결합 완료!")
print(f"- merged_data: {len(merged_data)} 레코드")
print(f"- export_subset: {len(export_subset)} 레코드")
print(f"- final_data: {len(final_data)} 레코드")

# 4. 결과 확인
if not final_data.empty:
    print(f"\n결합된 데이터 샘플 (상위 5개):")
    display_cols = ['ticker', 'date_month_end', 'market_cap_billions', 'revenue_ttm_shift', 'PSR_ttm', 'hs_code_6d', 'expDlr']
    available_cols = [col for col in display_cols if col in final_data.columns]
    print(final_data[available_cols].head().to_string(index=False))

    # 결측치 확인
    print(f"\n결측치 현황:")
    null_counts = final_data.isnull().sum()
    for col in ['hs_code_6d', 'expDlr']:
        if col in null_counts.index:
            print(f"- {col}: {null_counts[col]}개")

    # expDlr이 있는 데이터 개수
    if 'expDlr' in final_data.columns:
        valid_export = final_data['expDlr'].notna().sum()
        print(f"- 수출 데이터 매칭: {valid_export}개/{len(final_data)}개")

else:
    print("결합 실패")

print(f"\nfinal_data 변수로 접근 가능합니다.")

데이터프레임 결합 시작...
수출 데이터: 165개 레코드
수출 데이터 월말 변환 완료
데이터 결합 완료!
- merged_data: 185 레코드
- export_subset: 165 레코드
- final_data: 165 레코드

결합된 데이터 샘플 (상위 5개):
ticker date_month_end  market_cap_billions  revenue_ttm_shift  PSR_ttm hs_code_6d       expDlr
   MMM     2013-01-31                58.10             29.606 1.962440     851762 1233740000.0
   MMM     2013-02-28                60.09             29.904 2.009430     851762 1178980000.0
   MMM     2013-03-31                61.42             29.904 2.053906     851762 1378080000.0
   MMM     2013-04-30                60.25             29.904 2.014781     851762 1284210000.0
   MMM     2013-05-31                63.45             30.052 2.111340     851762 1275680000.0

결측치 현황:
- hs_code_6d: 0개
- expDlr: 0개
- 수출 데이터 매칭: 165개/165개

final_data 변수로 접근 가능합니다.


In [53]:
final_data

,date_month_end,hs_code_6d,expDlr,ticker,market_cap_billions,revenue_billions,revenue_ttm_billions,revenue_ttm_shift,PSR_ttm
0,2013-01-31,851762,1.233740e+09,MMM,58.10,7.39,29.904,29.606,1.962440
1,2013-02-28,851762,1.178980e+09,MMM,60.09,7.39,29.904,29.904,2.009430
2,2013-03-31,851762,1.378080e+09,MMM,61.42,7.63,30.052,29.904,2.053906
3,2013-04-30,851762,1.284210e+09,MMM,60.25,7.63,30.052,29.904,2.014781
4,2013-05-31,851762,1.275680e+09,MMM,63.45,7.63,30.052,30.052,2.111340
...,...,...,...,...,...,...,...,...,...
160,2026-05-31,851762,2.206470e+09,NaN,NaN,NaN,NaN,NaN,NaN
161,2026-06-30,851762,2.299530e+09,NaN,NaN,NaN,NaN,NaN,NaN
162,2026-07-31,851762,2.385440e+09,NaN,NaN,NaN,NaN,NaN,NaN
163,2026-08-31,851762,2.394650e+09,NaN,NaN,NaN,NaN,NaN,NaN


In [51]:
latest_export_data

,hs_code_6d,date,expDlr,forecast,quarter,input_date,forecast_flag,created_at
0,851762,2013-01-31,1.233740e+09,0,2013Q1,2025-09-02,0,2025-09-02 17:14:53
1,851762,2013-02-28,1.178980e+09,0,2013Q1,2025-09-02,0,2025-09-02 17:14:53
2,851762,2013-03-31,1.378080e+09,0,2013Q1,2025-09-02,0,2025-09-02 17:14:53
3,851762,2013-04-30,1.284210e+09,0,2013Q2,2025-09-02,0,2025-09-02 17:14:53
4,851762,2013-05-31,1.275680e+09,0,2013Q2,2025-09-02,0,2025-09-02 17:14:53
...,...,...,...,...,...,...,...,...
160,851762,2026-05-31,2.206470e+09,1,2026Q2,2025-09-02,1,2025-09-02 17:14:53
161,851762,2026-06-30,2.299530e+09,1,2026Q2,2025-09-02,1,2025-09-02 17:14:53
162,851762,2026-07-31,2.385440e+09,1,2026Q3,2025-09-02,1,2025-09-02 17:14:53
163,851762,2026-08-31,2.394650e+09,1,2026Q3,2025-09-02,1,2025-09-02 17:14:53


In [22]:
revenue_df_with_ttm

,ticker,date,calendar_year,period,revenue,revenue_billions,gross_profit,gross_margin,date_month_end,revenue_ttm,revenue_ttm_billions
159,MMM,1985-09-30,1985,Q3,2023000000,2.02,2023000000,100.00,1985-09-30,2.023000e+09,2.023
158,MMM,1985-12-31,1985,Q4,1963000000,1.96,1963000000,100.00,1985-12-31,3.986000e+09,3.986
157,MMM,1986-03-31,1986,Q1,2069000000,2.07,2069000000,100.00,1986-03-31,6.055000e+09,6.055
156,MMM,1986-06-30,1986,Q2,2185000000,2.19,2185000000,100.00,1986-06-30,8.240000e+09,8.240
155,MMM,1986-09-30,1986,Q3,2236000000,2.24,2236000000,100.00,1986-09-30,8.453000e+09,8.453
...,...,...,...,...,...,...,...,...,...,...,...
4,MMM,2024-06-30,2024,Q2,6255000000,6.25,2656000000,42.46,2024-06-30,2.454300e+10,24.543
3,MMM,2024-09-30,2024,Q3,6294000000,6.29,2643000000,41.99,2024-09-30,2.456700e+10,24.567
2,MMM,2024-12-31,2024,Q4,6010000000,6.01,2272000000,37.80,2024-12-31,2.457500e+10,24.575
1,MMM,2025-03-31,2025,Q1,5954000000,5.95,2436000000,40.91,2025-03-31,2.451300e+10,24.513
